# Person 6 — Gradient Boosting & Deployment Reporting Pipeline

Pipeline Responsibility: Web App Deployment & Benchmarking Report  
Model Assignment: Gradient Boosting  

This notebook loads the full labelled dataset from `data/raw`, trains Gradient Boosting, benchmarks latency, aggregates peer metrics from each model `outputs/` folder, and writes results to `outputs/`.


In [ ]:
from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix,
    classification_report, f1_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib

MODEL_FOLDER = "gradient_boosting"
HERE = Path.cwd().resolve()
ROOT = next(
    (
        p
        for p in (HERE, *HERE.parents)
        if (p / "data" / "raw").is_dir() and (p / "Basil_Leaf_ML_Workflow.ipynb").exists()
    ),
    None,
)
if ROOT is None:
    ROOT = next((p for p in (HERE, *HERE.parents) if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find project root containing data/raw.")

MODEL_DIR = ROOT / "parts" / MODEL_FOLDER
OUTPUT_DIR = MODEL_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "parts"))
from _pipeline import CLASSES, FEATURE_VERSION, SEED, prepare_dataset  # noqa: E402

print("Python:", sys.executable)
print("Project root:", ROOT)
print("Model folder:", MODEL_DIR)
print("Outputs:", OUTPUT_DIR)

data = prepare_dataset(ROOT)
X_tr, y_tr = data["X_tr"], data["y_tr"]
X_te, y_te = data["X_te"], data["y_te"]
manifest = data["manifest"]
print(f"Unique images: {len(manifest)} | train: {len(X_tr)} | test: {len(X_te)}")
print("Class counts:", data["audit"]["class_counts"])
print("Dataset complete listed counts:", not data["audit"]["download_coverage"]["partial_dataset"])
print("Feature version:", FEATURE_VERSION)


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

print("--- Person 6: Gradient Boosting ---")
gb_model = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=4,
    random_state=SEED,
)
start_time = time.perf_counter()
gb_model.fit(X_tr, y_tr)
fit_time = time.perf_counter() - start_time
preds = gb_model.predict(X_te)
acc = accuracy_score(y_te, preds)
p, r, f1, _ = precision_recall_fscore_support(y_te, preds, average="macro", zero_division=0)
cm = confusion_matrix(y_te, preds, labels=CLASSES)

bench_start = time.perf_counter()
for _ in range(100):
    _ = gb_model.predict(X_te[:10])
latency_ms = ((time.perf_counter() - bench_start) / 1000.0) * 1000.0
print(f"Accuracy: {acc:.4f} | Macro F1: {f1:.4f} | Latency: {latency_ms:.4f} ms/sample")
print("Confusion Matrix:\n", cm)

gb_metrics = {
    "model_name": "Gradient Boosting",
    "pipeline_stage": "Deployment & Benchmark Reporting",
    "accuracy": float(acc),
    "macro_f1": float(f1),
    "precision": float(p),
    "recall": float(r),
    "fit_time_seconds": float(fit_time),
    "latency_ms_per_sample": float(latency_ms),
    "confusion_matrix": cm.tolist(),
    "n_train": int(len(X_tr)),
    "n_test": int(len(X_te)),
    "classes": CLASSES,
    "feature_version": FEATURE_VERSION,
}
(OUTPUT_DIR / "gradient_boosting_metrics.json").write_text(json.dumps(gb_metrics, indent=2), encoding="utf-8")
joblib.dump(gb_model, OUTPUT_DIR / "gradient_boosting_model.joblib")

model_summaries = []
peer_files = [
    ROOT / "parts" / "logistic_regression" / "outputs" / "logistic_regression_metrics.json",
    ROOT / "parts" / "svm" / "outputs" / "svm_metrics.json",
    ROOT / "parts" / "knn" / "outputs" / "knn_metrics.json",
    ROOT / "parts" / "decision_tree" / "outputs" / "decision_tree_metrics.json",
    ROOT / "parts" / "random_forest" / "outputs" / "random_forest_metrics.json",
]
for path in peer_files:
    if path.exists():
        m = json.loads(path.read_text(encoding="utf-8"))
        model_summaries.append({
            "Model": m["model_name"],
            "Stage": m["pipeline_stage"],
            "Accuracy": m["accuracy"],
            "Macro_F1": m["macro_f1"],
            "Fit_Time_s": m["fit_time_seconds"],
        })
model_summaries.append({
    "Model": "Gradient Boosting",
    "Stage": "Deployment & Benchmark Reporting",
    "Accuracy": float(acc),
    "Macro_F1": float(f1),
    "Fit_Time_s": float(fit_time),
})
df_summary = pd.DataFrame(model_summaries)
df_summary.to_csv(OUTPUT_DIR / "model_comparison_6_members.csv", index=False)
print(df_summary.to_string(index=False))
print("Saved outputs to:", OUTPUT_DIR)
